# LLM Evaluation Pipeline – Separat vs. Kombiniert
Vergleicht zwei Prompt-Strategien über mehrere Modelle:
- **Separat**: drei einzelne Prompts pro Aufgabe
- **Kombiniert**: ein Prompt für alle drei Aufgaben

Metriken: Accuracy, Precision, Recall, F1-Score

## 1. Installation

In [ ]:
# Erste Zelle im Colab-Notebook — GPU prüfen
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import userdata
import os
import pathlib as path
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
"""
key_path = Path("/home/theo/PycharmProjects/Masterprojekt-Chatbots/API_KEYS/openai_key.txt")
if key_path.exists():
    os.environ["OPENAI_API_KEY"] = key_path.read_text().strip()
    print("OpenAI API Key gesetzt")
else:
    print("Key-Datei nicht gefunden")7
"""

In [ ]:
#%pip install -q langchain langchain-openai langchain-huggingface langchain-community
#%pip install -q transformers torch accelerate scikit-learn pandas pydantic tiktoken

In [ ]:
uploaded = files.upload()  # öffnet Datei-Dialog

import pandas as pd
df = pd.read_csv("gs_merged.csv")

## 2. Datensatz laden

In [ ]:
import pandas as pd
from pathlib import Path

repo = Path(".").resolve().parent
df   = pd.read_csv(repo / "data/processed/control/gs_merged.csv")

TASK_MAP = {
    1: "Informationssuche und Verständnis",
    2: "Schreiben und Textarbeit",
    3: "Praktische Unterstützung und Strukturierung",
    4: "Technische und analytische Unterstützung",
    5: "Lernen und Prüfungsvorbereitung",
}
SENTIMENT_MAP = {1: "Unfreundlich", 2: "Neutral", 3: "Freundlich"}
CRITICAL_MAP  = {0: "Nein", 1: "Ja"}

df_chats = (
    df.groupby("chat_id")
    .agg(
        text            = ("content", lambda x: "\n".join(
                            f"User_Nachricht_{i+1}: {msg}"
                            for i, msg in enumerate(x))),
        label_task      = ("task",      "first"),
        label_sentiment = ("sentiment", "first"),
        label_critical  = ("critical",  "first"),
    )
    .reset_index()
)

df_chats["label_task"]      = df_chats["label_task"].map(TASK_MAP)
df_chats["label_sentiment"] = df_chats["label_sentiment"].map(SENTIMENT_MAP)
df_chats["label_critical"]  = df_chats["label_critical"].map(CRITICAL_MAP)

print(f"Chats geladen: {len(df_chats)}")
df_chats[["chat_id", "label_task", "label_sentiment", "label_critical"]].head()

## 3. API Key & Labels

## 4. Pydantic Schemas & Labels

In [ ]:
from pydantic import BaseModel, field_validator
from typing import List, Optional

LABELS_TASK = [
    "Informationssuche und Verständnis",
    "Schreiben und Textarbeit",
    "Praktische Unterstützung und Strukturierung",
    "Technische und analytische Unterstützung",
    "Lernen und Prüfungsvorbereitung",
]
LABELS_SENTIMENT = ["Unfreundlich", "Neutral", "Freundlich"]
LABELS_CRITICAL  = ["Ja", "Nein"]

class ChatExample(BaseModel):
    text:             str
    label_task:       Optional[str] = None
    label_sentiment:  Optional[str] = None
    label_critical:   Optional[str] = None

    @field_validator("text")
    @classmethod
    def text_not_empty(cls, v):
        if not v.strip():
            raise ValueError("Text darf nicht leer sein")
        return v.strip()

# Beispiele aus DataFrame bauen
examples: List[ChatExample] = []
errors = []
for _, row in df_chats.iterrows():
    try:
        examples.append(ChatExample(
            text            = str(row["text"]),
            label_task      = str(row["label_task"]),
            label_sentiment = str(row["label_sentiment"]),
            label_critical  = str(row["label_critical"]),
        ))
    except Exception as e:
        errors.append({"row": row.to_dict(), "error": str(e)})

print(f"Gültige Beispiele: {len(examples)} | Übersprungen: {len(errors)}")

## 5. Kostenschätzung

In [ ]:
import tiktoken

PRICE_PER_1K = {
    "gpt-4o-mini":   {"input": 0.000150, "output": 0.000600},
    "gpt-3.5-turbo": {"input": 0.000500, "output": 0.001500},
    "gpt-4o":        {"input": 0.002500, "output": 0.010000},
}

def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

MODEL_NAMES    = ["gpt-4o-mini", "gpt-3.5-turbo"]  # <── anpassen
PROMPT_OVERHEAD_SEPARATE  = 200   # Token pro separatem Prompt (System + Template)
PROMPT_OVERHEAD_COMBINED  = 350   # Token für kombinierten Prompt (länger)
OUTPUT_TOKENS_SEPARATE    = 10    # pro Aufruf (ein Label)
OUTPUT_TOKENS_COMBINED    = 25    # pro Aufruf (drei Labels)

avg_text = sum(count_tokens(ex.text) for ex in examples) / len(examples)
n        = len(examples)

print(f"Chats: {n} | Ø Text-Token: {avg_text:.0f}")
print()
print(f"{'Modell':<20} {'Strategie':<12} {'Input-Token':>12} {'Output-Token':>13} {'Kosten (USD)':>13}")
print("-" * 72)

for model in MODEL_NAMES:
    if model not in PRICE_PER_1K:
        continue
    p = PRICE_PER_1K[model]

    # Separat: 3 Aufrufe pro Chat
    inp_sep = (avg_text + PROMPT_OVERHEAD_SEPARATE) * n * 3
    out_sep = OUTPUT_TOKENS_SEPARATE * n * 3
    cost_sep = (inp_sep / 1000) * p["input"] + (out_sep / 1000) * p["output"]

    # Kombiniert: 1 Aufruf pro Chat
    inp_com = (avg_text + PROMPT_OVERHEAD_COMBINED) * n
    out_com = OUTPUT_TOKENS_COMBINED * n
    cost_com = (inp_com / 1000) * p["input"] + (out_com / 1000) * p["output"]

    print(f"{model:<20} {'Separat':<12} {inp_sep:>12,.0f} {out_sep:>13,.0f} ${cost_sep:>12.4f}")
    print(f"{model:<20} {'Kombiniert':<12} {inp_com:>12,.0f} {out_com:>13,.0f} ${cost_com:>12.4f}")
    print(f"  → Ersparnis durch kombinierten Prompt: ${cost_sep - cost_com:.4f}")
    print()

## 6. Prompts

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Separate Prompts ──────────────────────────────────────────────────────────
prompt_task = ChatPromptTemplate.from_messages([
    ("system", """Du bist ein Klassifikationssystem. Analysiere das folgende Chat-Log und bestimme,
welcher Aufgabentyp vorliegt.

Aufgabentypen:
(1) Informationssuche und Verständnis – z.B. Literaturhinweise, Begriffe erklären, Verständnisfragen
(2) Schreiben und Textarbeit – z.B. Feedback, Gliederungen, Zusammenfassungen, Übersetzungen
(3) Praktische Unterstützung und Strukturierung – z.B. Ideen entwickeln, Projekte strukturieren, Planung
(4) Technische und analytische Unterstützung – z.B. Code, Datenanalyse, Statistik (R/SPSS/Stata)
(5) Lernen und Prüfungsvorbereitung – z.B. Prüfungsvorbereitung, Lernfragen, Karteikarten

Antworte NUR mit EINEM der folgenden Labels (exakt so):
{labels}"""),
    ("human", "Chat-Log:\n{text}\n\nLabel:"),
]).partial(labels=" | ".join(LABELS_TASK))

prompt_sentiment = ChatPromptTemplate.from_messages([
    ("system", """Du bist ein Klassifikationssystem. Analysiere den Ton des folgenden Chat-Logs.

Bewerte den Ton als:
- Unfreundlich: abweisend, fordernd, ungeduldig
- Neutral: sachlich, nüchtern, aufgabenorientiert
- Freundlich: höflich, wertschätzend, kooperativ

Antworte NUR mit EINEM der folgenden Labels (exakt so):
{labels}"""),
    ("human", "Chat-Log:\n{text}\n\nLabel:"),
]).partial(labels=" | ".join(LABELS_SENTIMENT))

prompt_critical = ChatPromptTemplate.from_messages([
    ("system", """Du bist ein Klassifikationssystem. Analysiere das folgende Chat-Log.

Fragt die Person ChatGPT nach Hinweisen zur Überprüfung der Antwort,
z.B. Quellen, Belege, Gegenargumente oder einer Prüfung auf mögliche Fehler?

Antworte NUR mit EINEM der folgenden Labels (exakt so):
{labels}"""),
    ("human", "Chat-Log:\n{text}\n\nLabel:"),
]).partial(labels=" | ".join(LABELS_CRITICAL))

# ── Kombinierter Prompt ───────────────────────────────────────────────────────
prompt_combined = ChatPromptTemplate.from_messages([
    ("system", """Du bist ein Klassifikationssystem für akademische Chat-Logs.
Analysiere das folgende Chat-Log und klassifiziere es nach drei Kriterien.

KRITERIUM 1 – Aufgabentyp:
(1) Informationssuche und Verständnis – z.B. Literaturhinweise, Begriffe erklären
(2) Schreiben und Textarbeit – z.B. Feedback, Gliederungen, Übersetzungen
(3) Praktische Unterstützung und Strukturierung – z.B. Ideen entwickeln, Planung
(4) Technische und analytische Unterstützung – z.B. Code, Datenanalyse, R/SPSS
(5) Lernen und Prüfungsvorbereitung – z.B. Prüfungsvorbereitung, Karteikarten

KRITERIUM 2 – Ton/Sentiment:
- Unfreundlich: abweisend, fordernd, ungeduldig
- Neutral: sachlich, nüchtern, aufgabenorientiert
- Freundlich: höflich, wertschätzend, kooperativ

KRITERIUM 3 – Kritische Überprüfung:
Fragt die Person nach Quellen, Belegen, Gegenargumenten oder Fehlerprüfung?
- Ja
- Nein

Antworte NUR in diesem exakten Format (drei Zeilen, nichts anderes):
TASK: <Label>
SENTIMENT: <Label>
CRITICAL: <Label>"""),
    ("human", "Chat-Log:\n{text}"),
])

print("Prompts definiert")

In [ ]:
pip install huggingface-hub llama-cpp-python

# Phi-3-mini (~2.4 GB)
huggingface-cli download microsoft/Phi-3-mini-4k-instruct-gguf \
  Phi-3-mini-4k-instruct-q4.gguf --local-dir ./models

# Mistral-7B (~4.4 GB)
huggingface-cli download TheBloke/Mistral-7B-Instruct-v0.3-GGUF \
  mistral-7b-instruct-v0.3.Q4_K_M.gguf --local-dir ./models

# LLaMA-3-8B (~5.0 GB)  ← braucht HuggingFace-Account + Lizenz-Zustimmung
huggingface-cli download meta-llama/Meta-Llama-3-8B-Instruct-GGUF \
  Meta-Llama-3-8B-Instruct.Q4_K_M.gguf --local-dir ./models

## 7. Modelle laden

In [ ]:
import torch
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

def load_local_llm(model_id: str, max_new_tokens: int = 32) -> HuggingFacePipeline:
    print(f"GPU verfügbar: {torch.cuda.is_available()}")  # sollte True sein
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,  # float16 auf GPU
        device_map="auto",          # landet automatisch auf der T4
    )
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
    )
    return HuggingFacePipeline(pipeline=pipe)

def make_chain(llm, prompt):
    return prompt | llm | StrOutputParser()

llm_registry = {
    #openAI_api
    "gpt-4o-mini":   ChatOpenAI(model="gpt-4o-mini",        temperature=0),
    "gpt-4.1":       ChatOpenAI(model="gpt-4.1",            temperature=0),
    "gpt-5.4-mini":  ChatOpenAI(model="gpt-5.4-mini",       temperature=0),

    ##local models
    "phi-3-mini":   load_local_llm("microsoft/Phi-3-mini-4k-instruct"),
    "mistral-7b":   load_local_llm("mistralai/Mistral-7B-Instruct-v0.3"),
    "llama-3-8b":   load_local_llm("meta-llama/Meta-Llama-3-8B-Instruct"),
}

print(f" Modelle bereit: {list(llm_registry.keys())}")

## 8. Evaluierungs-Funktionen

In [ ]:
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

def clean_label(raw: str, valid_labels: List[str]) -> str:
    raw = raw.strip()
    for label in valid_labels:               # exakter Match
        if label.lower() == raw.lower():
            return label
    for label in valid_labels:               # Teilstring-Match
        if label.lower() in raw.lower():
            return label
    return "unknown"

def compute_metrics(preds: List[str], golds: List[str], labels: List[str]) -> dict:
    valid = [(p, g) for p, g in zip(preds, golds) if p != "unknown"]
    if not valid:
        return {"error": "Keine gültigen Predictions"}
    p_clean, g_clean = zip(*valid)
    return {
        "accuracy":        round(accuracy_score(g_clean, p_clean), 4),
        "precision_macro": round(precision_score(g_clean, p_clean, average="macro", labels=labels, zero_division=0), 4),
        "recall_macro":    round(recall_score(g_clean, p_clean, average="macro",    labels=labels, zero_division=0), 4),
        "f1_macro":        round(f1_score(g_clean, p_clean, average="macro",        labels=labels, zero_division=0), 4),
        "unknown_rate":    round(1 - len(valid) / len(preds), 4),
        "n_samples":       len(preds),
        "report":          classification_report(g_clean, p_clean, labels=labels, zero_division=0),
    }

# Separat: eine Aufgabe pro Aufruf
def run_separate(llm, examples: List[ChatExample]) -> dict:

    TASKS = [
        {"name": "task_type", "prompt": prompt_task,      "labels": LABELS_TASK,      "gold_field": "label_task"},
        {"name": "sentiment",  "prompt": prompt_sentiment, "labels": LABELS_SENTIMENT, "gold_field": "label_sentiment"},
        {"name": "critical",   "prompt": prompt_critical,  "labels": LABELS_CRITICAL,  "gold_field": "label_critical"},
    ]
    results = {}
    total_latency = []

    for task in TASKS:
        print(f"    Aufgabe: {task['name']}")
        chain = make_chain(llm, task["prompt"])
        preds, golds, latencies = [], [], []

        for i, ex in enumerate(examples):
            gold = getattr(ex, task["gold_field"])
            if gold is None:
                continue
            start = time.time()
            try:
                raw  = chain.invoke({"text": ex.text})
                pred = clean_label(raw, task["labels"])
            except Exception as e:
                print(f"      Fehler bei Beispiel {i}: {e}")
                pred = "unknown"
            latencies.append(time.time() - start)
            preds.append(pred)
            golds.append(gold)

        metrics = compute_metrics(preds, golds, task["labels"])
        metrics["avg_latency_s"] = round(sum(latencies) / len(latencies), 3) if latencies else 0
        results[task["name"]]    = metrics
        total_latency.extend(latencies)
        print(f"      Accuracy: {metrics.get('accuracy')}  F1: {metrics.get('f1_macro')}")

    results["total_avg_latency_s"] = round(sum(total_latency) / len(total_latency), 3)
    return results

#Kombiniert: alle drei Aufgaben in einem Aufruf
def parse_combined(raw: str) -> dict:
    result = {"task": "unknown", "sentiment": "unknown", "critical": "unknown"}
    for line in raw.strip().splitlines():
        line = line.strip()
        if line.startswith("TASK:"):
            result["task"]      = clean_label(line.replace("TASK:", "").strip(), LABELS_TASK)
        elif line.startswith("SENTIMENT:"):
            result["sentiment"] = clean_label(line.replace("SENTIMENT:", "").strip(), LABELS_SENTIMENT)
        elif line.startswith("CRITICAL:"):
            result["critical"]  = clean_label(line.replace("CRITICAL:", "").strip(), LABELS_CRITICAL)
    return result

def run_combined(llm, examples: List[ChatExample]) -> dict:
    """Führt einen kombinierten Prompt für alle drei Aufgaben aus."""
    chain = make_chain(llm, prompt_combined)
    preds_task, preds_sentiment, preds_critical = [], [], []
    golds_task, golds_sentiment, golds_critical = [], [], []
    latencies = []

    for i, ex in enumerate(examples):
        start = time.time()
        try:
            raw    = chain.invoke({"text": ex.text})
            parsed = parse_combined(raw)
        except Exception as e:
            print(f"    Fehler bei Beispiel {i}: {e}")
            parsed = {"task": "unknown", "sentiment": "unknown", "critical": "unknown"}

        latencies.append(time.time() - start)
        preds_task.append(parsed["task"])
        preds_sentiment.append(parsed["sentiment"])
        preds_critical.append(parsed["critical"])
        golds_task.append(ex.label_task)
        golds_sentiment.append(ex.label_sentiment)
        golds_critical.append(ex.label_critical)

        if (i + 1) % 5 == 0:
            print(f"    {i+1}/{len(examples)} verarbeitet ...")

    avg_lat = round(sum(latencies) / len(latencies), 3)
    return {
        "task_type": {**compute_metrics(preds_task,      golds_task,      LABELS_TASK),      "avg_latency_s": avg_lat},
        "sentiment": {**compute_metrics(preds_sentiment, golds_sentiment, LABELS_SENTIMENT), "avg_latency_s": avg_lat},
        "critical":  {**compute_metrics(preds_critical,  golds_critical,  LABELS_CRITICAL),  "avg_latency_s": avg_lat},
        "total_avg_latency_s": avg_lat,
    }

print("✅ Funktionen definiert")

## 9. Evaluierung ausführen

In [ ]:
# Ergebnis-Struktur:
# all_results[model_name]["separate" | "combined"][task_name] = metrics
all_results = {}

for model_name, llm in llm_registry.items():
    print(f"\n{'='*60}")
    print(f"Modell: {model_name}")
    print(f"{'='*60}")
    all_results[model_name] = {}

    print("\n  ── Strategie: SEPARAT ──")
    all_results[model_name]["separate"] = run_separate(llm, examples)

    print("\n  ── Strategie: KOMBINIERT ──")
    all_results[model_name]["combined"] = run_combined(llm, examples)

print("\n✅ Evaluierung abgeschlossen")

## 10. Ergebnisse & Vergleich

In [ ]:
import pandas as pd

TASK_NAMES = ["task_type", "sentiment", "critical"]
rows = []

for model_name, strategies in all_results.items():
    for strategy, tasks in strategies.items():
        for task_name in TASK_NAMES:
            m = tasks.get(task_name, {})
            rows.append({
                "Modell":       model_name,
                "Strategie":    strategy,
                "Aufgabe":      task_name,
                "Accuracy":     m.get("accuracy"),
                "Precision":    m.get("precision_macro"),
                "Recall":       m.get("recall_macro"),
                "F1 (macro)":   m.get("f1_macro"),
                "Unknown-Rate": m.get("unknown_rate"),
                "Latenz (s)":   m.get("avg_latency_s"),
            })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(["Modell", "Aufgabe", "Strategie"])
summary_df

In [ ]:
#F1-Differenz: Separat vs. Kombiniert
print(f"{'Modell':<20} {'Aufgabe':<12} {'F1 Separat':>11} {'F1 Kombiniert':>14} {'Δ F1':>8}")
print("-" * 70)

for model_name, strategies in all_results.items():
    for task_name in TASK_NAMES:
        f1_sep = strategies["separate"].get(task_name, {}).get("f1_macro", None)
        f1_com = strategies["combined"].get(task_name, {}).get("f1_macro", None)
        if f1_sep is not None and f1_com is not None:
            delta = f1_com - f1_sep
            marker = "▲" if delta > 0 else ("▼" if delta < 0 else "=")
            print(f"{model_name:<20} {task_name:<12} {f1_sep:>11.4f} {f1_com:>14.4f} {marker}{abs(delta):>7.4f}")

In [ ]:
#Classification Reports
for model_name, strategies in all_results.items():
    for strategy, tasks in strategies.items():
        for task_name in TASK_NAMES:
            m = tasks.get(task_name, {})
            if "report" in m:
                print(f"\n{'='*60}")
                print(f"{model_name}  |  {strategy}  |  {task_name}")
                print(f"{'='*60}")
                print(m["report"])

In [ ]:
import json

summary_df.to_csv("eval_results_v3.csv", index=False)
print("eval_results_v3.csv gespeichert")

export = {
    model: {
        strategy: {
            task: {k: v for k, v in metrics.items() if k != "report"}
            for task, metrics in tasks.items()
            if isinstance(metrics, dict)
        }
        for strategy, tasks in strategies.items()
    }
    for model, strategies in all_results.items()
}
with open("eval_results_v3.json", "w", encoding="utf-8") as f:
    json.dump(export, f, ensure_ascii=False, indent=2)
print("eval_results_v3.json gespeichert")